# GLA UNIVERSITY
### Department of Computer Engineering & Applications
## **ML Software Project: Music Genre Classifier & Explorer**

---
* **Dataset:** GTZAN Benchmark (1,000 Tracks across 10 Genres)
* **Acoustic Descriptors:** 58 Features (MFCCs 1-20, Spectral Centroid, Rolloff, Chroma STFT, RMS, ZCR, HPSS, Tempo)
* **Classifiers:** Random Forest (Baseline), Support Vector Machine (Comparison), Hybrid Soft-Voting Ensemble
---
### **Project Objectives:**
1. Ingest and preprocess the GTZAN audio feature dataset with missing value imputation.
2. Perform Exploratory Data Analysis (EDA) and detect acoustic feature outliers.
3. Train supervised multiclass classifiers for genre prediction.
4. Evaluate model performance using Accuracy, Macro-Precision, Macro-Recall, Macro-F1, and Confusion Matrices.
5. Export versioned model artifacts for the interactive web visualizer.



### **Step 1: Import Required Libraries**


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import joblib

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline



### **Step 2: Dataset Loading & Inspection**


In [ ]:
csv_path = 'features_30_sec_with_missing_values (1).csv'
df = pd.read_csv(csv_path)

print('Dataset Shape:', df.shape)
print('Columns Count:', len(df.columns))
print('Total Missing Values:', df.isnull().sum().sum())
df.head(5)



### **Step 3: Data Cleaning & Missing Value Imputation**


In [ ]:
# 1. Check and drop duplicate records
dup_count = df.duplicated().sum()
print(f'Duplicate Rows Found: {dup_count}')
df = df.drop_duplicates()

# 2. Impute numerical columns with Median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# 3. Impute categorical columns with Mode
cat_cols = df.select_dtypes(include=['object', 'string']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print('Total Missing Values After Imputation:', df.isnull().sum().sum())



### **Step 4: Outlier Detection via Interquartile Range (IQR)**


In [ ]:
outlier_records = []
for col in numeric_cols:
    if col != 'length':
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = ((df[col] < lower) | (df[col] > upper)).sum()
        if count > 0:
            outlier_records.append({'Feature': col, 'Outlier Count': count, 'Lower Bound': round(lower, 3), 'Upper Bound': round(upper, 3)})

outlier_df = pd.DataFrame(outlier_records).sort_values(by='Outlier Count', ascending=False)
print(f'Total features exhibiting outliers: {len(outlier_df)}')
outlier_df.head(10)



### **Step 5: Exploratory Data Analysis & Visualizations**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Genre Class Balance
df['label'].value_counts().plot(kind='bar', ax=axes[0], color='teal', edgecolor='black')
axes[0].set_title('Genre Class Distribution (100 samples each)')
axes[0].set_ylabel('Track Count')
axes[0].tick_params(axis='x', rotation=45)

# 2. Spectral Centroid distribution across Genres
sns.boxplot(x='label', y='spectral_centroid_mean', data=df, ax=axes[1], palette='tab10')
axes[1].set_title('Spectral Centroid Mean (Brightness) by Genre')
axes[1].set_ylabel('Hz')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()



### **Step 6: Feature Extraction & Train-Test Splitting**


In [ ]:
# Separate features and target
X = df.drop(columns=[c for c in ['label', 'filename'] if c in df.columns])
y = df['label']

# Encode target genres
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
classes = encoder.classes_
print('Classes (10):', list(classes))
print('Features count:', X.shape[1])

# Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

print(f'Train Shape: {X_train.shape}, Test Shape: {X_test.shape}')



### **Step 7: Model 1 - Random Forest Classifier (Baseline)**


In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=22,
    min_samples_split=3,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)
print(f'Random Forest Test Accuracy: {rf_acc * 100:.2f}%')
print('\nRandom Forest Classification Report:')
print(classification_report(y_test, rf_pred, target_names=classes))



### **Step 8: Model 2 - Support Vector Machine (Comparison Model)**


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm = SVC(
    C=4.0,
    kernel='rbf',
    gamma='scale',
    probability=True,
    random_state=42
)
svm.fit(X_train_scaled, y_train)

svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(f'SVM Test Accuracy: {svm_acc * 100:.2f}%')
print('\nSVM Classification Report:')
print(classification_report(y_test, svm_pred, target_names=classes))



### **Step 9: Model 3 - Hybrid Soft-Voting Ensemble (Optimal)**


In [ ]:
et = ExtraTreesClassifier(n_estimators=300, max_depth=25, random_state=42, n_jobs=-1)
et.fit(X_train, y_train)

# Blend posterior probabilities: 40% RF + 35% SVM + 25% ExtraTrees
rf_probs = rf.predict_proba(X_test)
svm_probs = svm.predict_proba(X_test_scaled)
et_probs = et.predict_proba(X_test)

ensemble_probs = (rf_probs * 0.40) + (svm_probs * 0.35) + (et_probs * 0.25)
ensemble_pred = np.argmax(ensemble_probs, axis=1)

ens_acc = accuracy_score(y_test, ensemble_pred)
print(f'Hybrid Ensemble Test Accuracy: {ens_acc * 100:.2f}%')
print('\nEnsemble Classification Report:')
print(classification_report(y_test, ensemble_pred, target_names=classes))



### **Step 10: Confusion Matrix Visualizations**


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Random Forest CM
cm_rf = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title(f'Random Forest Baseline (Accuracy: {rf_acc*100:.1f}%)')
axes[0].set_ylabel('True Genre')
axes[0].set_xlabel('Predicted Genre')

# Ensemble CM
cm_ens = confusion_matrix(y_test, ensemble_pred)
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Greens', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title(f'Hybrid Soft-Voting Ensemble (Accuracy: {ens_acc*100:.1f}%)')
axes[1].set_ylabel('True Genre')
axes[1].set_xlabel('Predicted Genre')

plt.tight_layout()
plt.show()



### **Step 11: Acoustic Feature Importance Analysis**


In [ ]:
importances = rf.feature_importances_
feat_names = X.columns
sorted_idx = np.argsort(importances)[::-1][:20]

plt.figure(figsize=(12, 6))
plt.barh([feat_names[i] for i in reversed(sorted_idx)], [importances[i] for i in reversed(sorted_idx)], color='purple', alpha=0.85)
plt.title('Top 20 Most Informative Audio Features (Gini Importance)')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.tight_layout()
plt.show()



### **Step 12: Model Persistence for Web Inference**


In [ ]:
models_dir = 'ml/models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf, os.path.join(models_dir, 'random_forest.joblib'))
joblib.dump(svm, os.path.join(models_dir, 'svm.joblib'))
joblib.dump(et, os.path.join(models_dir, 'extra_trees.joblib'))
joblib.dump(scaler, os.path.join(models_dir, 'scaler.joblib'))
joblib.dump(encoder, os.path.join(models_dir, 'label_encoder.joblib'))

with open(os.path.join(models_dir, 'feature_names.json'), 'w') as f:
    json.dump(list(X.columns), f, indent=2)

print('All model artifacts saved successfully in:', models_dir)

